# 01_bronze_ingest

Reads raw Binance kline CSVs out of the Unity Catalog Volume using an **explicit
schema** (Phase 2 Requirement 2), performs **idempotent MERGE** upserts into
Bronze (Requirement 3), quarantines schema-drift rows instead of crashing
(Requirement 3), and writes an audit log entry for every file processed
(Requirement 4).

In [ ]:
import uuid
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType, StringType, TimestampType
)
from pyspark.sql.functions import col, lit, current_timestamp, current_date
from delta.tables import DeltaTable

## Constants & explicit schemas (no inferSchema anywhere)

In [ ]:
CATALOG = "workspace"
SCHEMA = "cryptolens"
VOLUME_ROOT = "/Volumes/workspace/cryptolens/bronze_raw/l242529@lhr.nu.edu.pk"

BRONZE_KLINES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_klines"
BRONZE_EXCHANGE_INFO_TABLE = f"{CATALOG}.{SCHEMA}.bronze_exchange_info"
BRONZE_QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_quarantine"
DIM_SYMBOL_TABLE = f"{CATALOG}.{SCHEMA}.dim_symbol"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"

# Binance kline CSVs are headerless, 12 columns, fixed order. _corrupt_record
# is a 13th "virtual" column: PERMISSIVE mode populates it only for rows that
# fail to parse against this schema -- this is how schema drift / bad rows
# are caught without inferSchema and without crashing the whole file read.
RAW_KLINE_SCHEMA = StructType([
    StructField("open_time", LongType(), True),                      # epoch ms
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close_time", LongType(), True),                     # epoch ms
    StructField("quote_asset_volume", DoubleType(), True),
    StructField("number_of_trades", LongType(), True),
    StructField("taker_buy_base_asset_volume", DoubleType(), True),
    StructField("taker_buy_quote_asset_volume", DoubleType(), True),
    StructField("ignore", StringType(), True),                       # unused Binance field
    StructField("_corrupt_record", StringType(), True),              # PERMISSIVE-mode capture column
])

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),                 # 'Raw-to-Bronze' | 'Bronze-to-Silver'
    StructField("parameter_processed", StringType(), False),   # file path / symbol-interval-loadtype
    StructField("load_type", StringType(), False),             # 'full' | 'incremental'
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),                # 'SUCCESS' | 'FAILURE' | 'SKIPPED'
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
])

## Audit logging helper (Requirement 4)

In [ ]:
def log_execution(spark, layer, parameter_processed, load_type, start_time,
                   status, rows_inserted=0, rows_updated=0, error_message=None):
    """Append one row to pipeline_execution_logs. Called on every success,
    failure, and skip path below."""
    end_time = datetime.utcnow()
    row = [(
        str(uuid.uuid4()), layer, parameter_processed, load_type,
        start_time, end_time, status,
        int(rows_inserted) if rows_inserted is not None else 0,
        int(rows_updated) if rows_updated is not None else 0,
        error_message,
    )]
    log_df = spark.createDataFrame(row, schema=LOG_SCHEMA)
    log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)

## Parameterized path helpers (Requirement 3 — no hardcoded "today")

In [ ]:
import re

def build_volume_path(symbol, interval, load_type, year, month, day=None):
    """Builds the Volume path for a given symbol/date/load_type. Pass any
    historical year/month/day to backfill -- nothing here assumes 'today'."""
    month = f"{int(month):02d}"
    if load_type == "full":
        return f"{VOLUME_ROOT}/full_load/{symbol}-{interval}-{year}-{month}.csv"
    elif load_type == "incremental":
        if not day:
            raise ValueError("day is required for load_type='incremental'")
        day = f"{int(day):02d}"
        return f"{VOLUME_ROOT}/incremental_load/{symbol}-{interval}-{year}-{month}-{day}.csv"
    else:
        raise ValueError(f"Unknown load_type: {load_type}")


def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def discover_symbols(interval, load_type, year, month, day=None):
    """Scans the Volume and returns every symbol that actually has a file
    for this exact date/load_type -- no manual symbol list, no dependency on
    dim_symbol (which doesn't exist until Silver has run at least once)."""
    folder = f"{VOLUME_ROOT}/{'full_load' if load_type == 'full' else 'incremental_load'}"
    try:
        files = dbutils.fs.ls(folder)
    except Exception:
        return []

    month_padded = f"{int(month):02d}"
    if load_type == "full":
        suffix = f"-{interval}-{year}-{month_padded}.csv"
    else:
        day_padded = f"{int(day):02d}"
        suffix = f"-{interval}-{year}-{month_padded}-{day_padded}.csv"

    symbols = []
    for f in files:
        if f.name.endswith(suffix):
            match = re.match(r"([A-Z0-9]+)-", f.name)
            if match:
                symbols.append(match.group(1))
    return sorted(symbols)

## Parameters
Re-run this notebook for ANY historical month/day by changing the widgets
below -- nothing in the code assumes "today". Symbols are auto-discovered
from whatever files are actually present in the Volume for the chosen
date/load_type (see discover_symbols above) -- no manual symbol list
needed, and no dependency on dim_symbol existing yet.

In [ ]:
dbutils.widgets.text("interval", "1m")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"])
dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "08")
dbutils.widgets.text("day", "")              # required only when load_type = incremental

interval = dbutils.widgets.get("interval")
load_type = dbutils.widgets.get("load_type")
year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
day = dbutils.widgets.get("day") or None

## Bronze ingestion — one kline file in, idempotent MERGE out

In [ ]:
def ingest_kline_file(spark, symbol, interval, load_type, file_path):
    start_time = datetime.utcnow()
    try:
        if not file_exists(file_path):
            log_execution(spark, "Raw-to-Bronze", file_path, load_type, start_time,
                          "SKIPPED", error_message="File not found in Volume")
            print(f"[SKIP] {file_path} not found")
            return

        # Strict schema-on-read: explicit StructType, PERMISSIVE mode routes
        # any row that doesn't match the schema into _corrupt_record instead
        # of failing the whole read (Requirement 2 + schema-drift handling).
        raw_df = (spark.read
                  .option("header", False)
                  .option("mode", "PERMISSIVE")
                  .option("columnNameOfCorruptRecord", "_corrupt_record")
                  .schema(RAW_KLINE_SCHEMA)
                  .csv(file_path))

        good_df = raw_df.filter(col("_corrupt_record").isNull()).drop("_corrupt_record")
        bad_df = raw_df.filter(col("_corrupt_record").isNotNull())

        bad_count = bad_df.count()
        if bad_count > 0:
            (bad_df
                .withColumn("source_file", lit(file_path))
                .withColumn("load_timestamp", current_timestamp())
                .write.format("delta").mode("append")
                .option("mergeSchema", "true")   # tolerate drift in the quarantine table itself
                .saveAsTable(BRONZE_QUARANTINE_TABLE))
            print(f"[WARN] {bad_count} malformed rows quarantined from {file_path}")

        enriched_df = (good_df
            .withColumn("symbol", lit(symbol))
            .withColumn("interval", lit(interval))
            .withColumn("load_type", lit(load_type))
            .withColumn("source_file", lit(file_path))
            .withColumn("ingestion_date", current_date())
            .withColumn("load_timestamp", current_timestamp()))  # Requirement 2: load_timestamp on every record

        # Idempotent upsert (Requirement 3): natural key = symbol + interval +
        # load_type + open_time. Re-running this notebook on the same file
        # inserts zero new rows the second time.
        if spark.catalog.tableExists(BRONZE_KLINES_TABLE):
            target = DeltaTable.forName(spark, BRONZE_KLINES_TABLE)
            (target.alias("t")
                .merge(enriched_df.alias("s"),
                       "t.symbol = s.symbol AND t.interval = s.interval "
                       "AND t.load_type = s.load_type AND t.open_time = s.open_time")
                .whenNotMatchedInsertAll()
                .execute())
        else:
            (enriched_df.write.format("delta")
                .option("mergeSchema", "true")
                .saveAsTable(BRONZE_KLINES_TABLE))

        row_count = enriched_df.count()
        log_execution(spark, "Raw-to-Bronze", file_path, load_type, start_time,
                      "SUCCESS", rows_inserted=row_count)
        print(f"[OK] {file_path} -> {row_count} rows merged into bronze_klines")

    except Exception as e:
        log_execution(spark, "Raw-to-Bronze", file_path, load_type, start_time,
                      "FAILURE", error_message=str(e))
        print(f"[FAIL] {file_path}: {e}")
        raise

## Driver loop — auto-discovers and processes every symbol with a file for this date/load_type

In [ ]:
symbols = discover_symbols(interval, load_type, year, month, day)

if not symbols:
    print(f"[WARN] No files found for load_type={load_type}, interval={interval}, "
          f"{year}-{month}" + (f"-{day}" if day else "") + " -- check the Volume path/date.")
else:
    print(f"Discovered {len(symbols)} symbol(s) with data for {year}-{month}" + (f"-{day}" if day else ""))

for sym in symbols:
    path = build_volume_path(sym, interval, load_type, year, month, day)
    ingest_kline_file(spark, sym, interval, load_type, path)

## exchangeInfo snapshot ingestion (feeds dim_symbol in Silver)
Landed as raw JSON text with explicit schema + load_timestamp, same pattern
as the kline files above. Run this cell whenever a fresh exchangeInfo JSON
file has been uploaded to the Volume (see the Phase 1 network-workaround
notes for how that file gets there).

In [ ]:
def ingest_exchange_info_snapshot(spark, file_path):
    start_time = datetime.utcnow()
    try:
        if not file_exists(file_path):
            log_execution(spark, "Raw-to-Bronze", file_path, "snapshot", start_time,
                          "SKIPPED", error_message="File not found in Volume")
            return

        # Land the whole file as a single raw_json string column -- Bronze
        # stays schema-agnostic for this source; flattening happens in Silver.
        raw_df = (spark.read.text(file_path)
                  .selectExpr("concat_ws('\\n', collect_list(value)) as raw_json")
                  .groupBy().agg({"raw_json": "max"})
                  .toDF("raw_json"))

        enriched_df = (raw_df
            .withColumn("source_file", lit(file_path))
            .withColumn("load_timestamp", current_timestamp()))

        (enriched_df.write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(BRONZE_EXCHANGE_INFO_TABLE))

        log_execution(spark, "Raw-to-Bronze", file_path, "snapshot", start_time,
                      "SUCCESS", rows_inserted=1)
        print(f"[OK] exchangeInfo snapshot landed from {file_path}")
    except Exception as e:
        log_execution(spark, "Raw-to-Bronze", file_path, "snapshot", start_time,
                      "FAILURE", error_message=str(e))
        raise

# Example call once the JSON snapshot file exists in the Volume:
# ingest_exchange_info_snapshot(spark, f"{VOLUME_ROOT}/exchange_info/exchangeInfo-2026-09-25.json")